# Notebook 15 — Neural Network (MLP + Entity Embeddings)

**Goal:** Train an MLP with entity embeddings on the 38-feature v2 set. Measure Spearman ρ vs LGBM NB12 OOF. Target: achieve meaningful architectural diversity (ρ < 0.90) that enables a stronger rank-average ensemble in NB17.

**Why MLP after NB14 confirmed GBM feature engineering is exhausted:**  
All GBMs (LGBM, CatBoost, ET) share the same 38 features and piecewise-constant decision boundaries → Spearman ρ ≥ 0.93 → they fail on the same hard laps. An MLP learns smooth, continuous boundaries and can model cross-feature interactions that GBMs regularize away. Entity embeddings for Driver/Race/Compound/Year give the network a learnable representation of categorical identity rather than relying on fold-averaged numeric encodings.

**Architecture:**
- Entity embeddings: `Driver` (dim=32), `Race` (dim=8), `Compound` (dim=3), `Year` (dim=2) → 45 embedding dims
- Continuous: all 38 FEAT_COLS (same as NB11/NB12) → 38 dims
- MLP input: 83 dims → Dense(256)+BN+ReLU+Drop(0.3) → Dense(128)+BN+ReLU+Drop(0.2) → Dense(64)+BN+ReLU+Drop(0.1) → Dense(1)
- Loss: `BCEWithLogitsLoss(pos_weight≈4.0)` · Optimizer: AdamW lr=1e-3 · Scheduler: CosineAnnealingLR
- Early stopping: patience=10 on val AUC · Max 100 epochs · Batch size 4096

**Inputs:** `train_features_v2.parquet`, `test_features_v2.parquet`, `fold_assignments.parquet`, `oof_predictions_nb12.parquet` (for ρ baseline)  
**Outputs:** `oof_predictions_nb15.parquet`, `test_predictions_nb15.parquet`, `models/mlp_fold{0-4}.pt`, `models/nb15_summary.pkl`

**Gate for NB17 ensemble inclusion:**
- Solo AUC ≥ 0.895 (minimum strength threshold)
- If ρ < 0.90: meaningful diversity → strong inclusion case
- If 0.90 ≤ ρ < 0.95: marginal diversity → include only if rank-avg Δ ≥ +0.001 on OOF
- If ρ ≥ 0.95: exclude (correlated, same errors as GBMs)

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr, rankdata
import pickle
import warnings
import time
from pathlib import Path

warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n_threads = torch.get_num_threads()
print(f'PyTorch   : {torch.__version__}')
print(f'Device    : {device}')
print(f'CPU threads: {n_threads}')

PyTorch   : 2.12.0+cpu
Device    : cpu
CPU threads: 8


In [2]:
# Robust project root detection — works from workspace root, notebooks/, or Kaggle
cwd = Path.cwd()
while cwd.name != 'predict-f1-pit-stops' and cwd.parent != cwd:
    cwd = cwd.parent
project_root  = cwd
data_dir      = project_root / 'data' / 'raw'
processed_dir = project_root / 'data' / 'processed'
models_dir    = project_root / 'models'
models_dir.mkdir(parents=True, exist_ok=True)

if not data_dir.exists():
    data_dir      = Path('/kaggle/input/playground-series-s6e5')
    processed_dir = Path('/kaggle/working')

print(f'Project root : {project_root}')
print(f'Processed dir: {processed_dir}')

train = pd.read_parquet(processed_dir / 'train_features_v2.parquet')
test  = pd.read_parquet(processed_dir / 'test_features_v2.parquet')
folds = pd.read_parquet(processed_dir / 'fold_assignments.parquet')

train = train.merge(folds, on='id', how='left')

print(f'Train: {train.shape}  |  Test: {test.shape}')
print(f'Class balance: {train["PitNextLap"].mean():.3f} positive rate')
print(f'Fold distribution:\n{train["fold"].value_counts().sort_index().to_string()}')

Project root : c:\Repos\predict-f1-pit-stops
Processed dir: c:\Repos\predict-f1-pit-stops\data\processed
Train: (439140, 53)  |  Test: (188165, 51)
Class balance: 0.199 positive rate
Fold distribution:
fold
0    88018
1    87444
2    88027
3    87854
4    87797


## Feature Set

Identical 38-feature set from NB11/NB12. All features are numeric — target encodings are computed fold-aware inside each CV fold (same `apply_target_encodings` function as NB11).

Additionally, `Driver`, `Race`, `Compound`, and `Year` are used as **categorical embedding inputs** (separate from the 38 continuous features). This gives the MLP both the numeric encoding signal AND a learnable embedding per category.

In [3]:
BASE_FEATURES = [
    'TyreLife_normalized_by_compound', 'TyreLife_sq',
    'is_wet_tyre', 'compound_ordinal',
    'laps_remaining', 'is_testing_session',
    'Stint', 'Position',
    'Cumulative_Degradation_winsorized', 'Degradation_rate', 'Degradation_acceleration',
    'LapTime_lag1', 'LapTime_lag2', 'LapTime_lag3',
    'LapTime_Delta_lag1', 'LapTime_Delta_lag2', 'LapTime_Delta_lag3',
    'LapTime_rolling_mean_3', 'LapTime_rolling_mean_5',
    'LapTime_rolling_std_3',  'LapTime_rolling_std_5',
    'Degradation_rolling_slope_3', 'Degradation_rolling_slope_5',
    'TyreLife_x_laps_remaining', 'Degradation_x_RaceProgress', 'Position_x_RaceProgress',
]
TARGET_ENC = ['Race_target_encoded', 'Driver_target_encoded', 'Driver_avg_stint_length']
TIER14_FEATS = [
    'TyreLife_x_compound_ordinal', 'Stint_x_compound_ordinal', 'TyreLife_x_cmpd_x_laps_rem',
    'prime_pit_window', 'prime_window_x_compound',
    'abs_position_change', 'pos_change_rolling_std_3',
    'PitStop_lag1', 'laps_to_driver_end',
]
FEAT_COLS = BASE_FEATURES + TARGET_ENC + TIER14_FEATS  # 38 total continuous inputs

CAT_COLS = ['Driver', 'Race', 'Compound', 'Year']  # entity embedding inputs

print(f'Continuous features : {len(FEAT_COLS)}')
print(f'Embedding categoricals: {CAT_COLS}')

non_enc = BASE_FEATURES + TIER14_FEATS
missing_train = [f for f in non_enc if f not in train.columns]
missing_test  = [f for f in non_enc if f not in test.columns]
assert not missing_train, f'Missing in train: {missing_train}'
assert not missing_test,  f'Missing in test: {missing_test}'
print('Feature presence check: OK')


def apply_target_encodings(train_df: pd.DataFrame, val_df: pd.DataFrame) -> pd.DataFrame:
    """Fold-aware target encodings. Must be called inside each CV fold."""
    global_mean  = train_df['PitNextLap'].mean()
    race_map     = train_df.groupby('Race')['PitNextLap'].mean()
    driver_map   = train_df.groupby('Driver')['PitNextLap'].mean()
    stint_map    = (
        train_df.groupby(['Driver', 'Race', 'Year', 'Stint'])['TyreLife']
        .max().reset_index().groupby('Driver')['TyreLife'].mean()
    )
    global_stint = stint_map.mean()

    out = val_df.copy()
    out['Race_target_encoded']     = out['Race'].map(race_map).fillna(global_mean)
    out['Driver_target_encoded']   = out['Driver'].map(driver_map).fillna(global_mean)
    out['Driver_avg_stint_length'] = out['Driver'].map(stint_map).fillna(global_stint)
    return out

print('apply_target_encodings defined.')

Continuous features : 38
Embedding categoricals: ['Driver', 'Race', 'Compound', 'Year']
Feature presence check: OK
apply_target_encodings defined.


In [4]:
# Label encoders for entity embedding inputs
# Fit on combined train+test so test inference never sees an unknown index.
le_driver   = LabelEncoder()
le_race     = LabelEncoder()
le_compound = LabelEncoder()
le_year     = LabelEncoder()

le_driver.fit(np.concatenate([train['Driver'].values, test['Driver'].values]))
le_race.fit(np.concatenate([train['Race'].values, test['Race'].values]))
le_compound.fit(np.concatenate([train['Compound'].values, test['Compound'].values]))
le_year.fit(np.concatenate([train['Year'].astype(str).values, test['Year'].astype(str).values]))

N_DRIVERS   = len(le_driver.classes_)
N_RACES     = len(le_race.classes_)
N_COMPOUNDS = len(le_compound.classes_)
N_YEARS     = len(le_year.classes_)

# Embedding dimensions — Driver is large (887 unique) so give it more capacity
EMB_DIM_DRIVER   = 32
EMB_DIM_RACE     = 8
EMB_DIM_COMPOUND = 3
EMB_DIM_YEAR     = 2
TOTAL_EMB = EMB_DIM_DRIVER + EMB_DIM_RACE + EMB_DIM_COMPOUND + EMB_DIM_YEAR
TOTAL_IN  = len(FEAT_COLS) + TOTAL_EMB

print(f'Cardinalities  : Driver={N_DRIVERS}, Race={N_RACES}, Compound={N_COMPOUNDS}, Year={N_YEARS}')
print(f'Embedding dims : driver={EMB_DIM_DRIVER}, race={EMB_DIM_RACE}, compound={EMB_DIM_COMPOUND}, year={EMB_DIM_YEAR}  → total={TOTAL_EMB}')
print(f'MLP input dim  : {len(FEAT_COLS)} continuous + {TOTAL_EMB} embed = {TOTAL_IN}')


def encode_cats(df: pd.DataFrame) -> dict:
    return {
        'driver':   le_driver.transform(df['Driver'].values).astype(np.int64),
        'race':     le_race.transform(df['Race'].values).astype(np.int64),
        'compound': le_compound.transform(df['Compound'].values).astype(np.int64),
        'year':     le_year.transform(df['Year'].astype(str).values).astype(np.int64),
    }

Cardinalities  : Driver=887, Race=26, Compound=5, Year=4
Embedding dims : driver=32, race=8, compound=3, year=2  → total=45
MLP input dim  : 38 continuous + 45 embed = 83


## MLP Architecture

```
Input (83) = [38 continuous (StandardScaled)] + [Driver_emb(32) | Race_emb(8) | Compound_emb(3) | Year_emb(2)]
    ↓
Linear(83 → 256) → BatchNorm1d → ReLU → Dropout(0.3)
    ↓
Linear(256 → 128) → BatchNorm1d → ReLU → Dropout(0.2)
    ↓
Linear(128 → 64) → BatchNorm1d → ReLU → Dropout(0.1)
    ↓
Linear(64 → 1)   [raw logit — BCEWithLogitsLoss during train, sigmoid at inference]
```

In [5]:
class PitStopMLP(nn.Module):
    def __init__(self, n_cont, n_drivers, n_races, n_compounds, n_years,
                 emb_dim_driver=32, emb_dim_race=8, emb_dim_compound=3, emb_dim_year=2):
        super().__init__()
        self.driver_emb   = nn.Embedding(n_drivers,   emb_dim_driver)
        self.race_emb     = nn.Embedding(n_races,     emb_dim_race)
        self.compound_emb = nn.Embedding(n_compounds, emb_dim_compound)
        self.year_emb     = nn.Embedding(n_years,     emb_dim_year)

        total_in = n_cont + emb_dim_driver + emb_dim_race + emb_dim_compound + emb_dim_year
        self.mlp = nn.Sequential(
            nn.Linear(total_in, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128),      nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 64),       nn.BatchNorm1d(64),  nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(64, 1),
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.01)

    def forward(self, x_cont, x_driver, x_race, x_compound, x_year):
        embs = torch.cat([
            self.driver_emb(x_driver),
            self.race_emb(x_race),
            self.compound_emb(x_compound),
            self.year_emb(x_year),
        ], dim=1)
        return self.mlp(torch.cat([x_cont, embs], dim=1)).squeeze(1)


def make_model():
    return PitStopMLP(
        n_cont=len(FEAT_COLS), n_drivers=N_DRIVERS, n_races=N_RACES,
        n_compounds=N_COMPOUNDS, n_years=N_YEARS,
        emb_dim_driver=EMB_DIM_DRIVER, emb_dim_race=EMB_DIM_RACE,
        emb_dim_compound=EMB_DIM_COMPOUND, emb_dim_year=EMB_DIM_YEAR,
    ).to(device)


_m = make_model()
total_params = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f'Total parameters: {total_params:,}')
print(f'  Embedding params: {N_DRIVERS*EMB_DIM_DRIVER + N_RACES*EMB_DIM_RACE + N_COMPOUNDS*EMB_DIM_COMPOUND + N_YEARS*EMB_DIM_YEAR:,}')
del _m

Total parameters: 92,232
  Embedding params: 28,615


In [6]:
# Training hyperparameters
BATCH_SIZE   = 4096
MAX_EPOCHS   = 100
PATIENCE     = 10
LR           = 1e-3
WEIGHT_DECAY = 1e-4

pos_rate       = train['PitNextLap'].mean()
POS_WEIGHT_VAL = (1.0 - pos_rate) / pos_rate  # ~4.0 for 20% positive rate

print(f'Batch size  : {BATCH_SIZE}')
print(f'Max epochs  : {MAX_EPOCHS}  (early stop patience={PATIENCE})')
print(f'LR / WD     : {LR} / {WEIGHT_DECAY}')
print(f'Positive rate: {pos_rate:.3f}  →  pos_weight: {POS_WEIGHT_VAL:.2f}')


def df_to_tensors(df_enc: pd.DataFrame, cat_codes: dict,
                  scaler: StandardScaler = None, fit_scaler: bool = False):
    """Convert encoded DataFrame + cat codes to torch tensors. Returns tensors + scaler."""
    X_cont = df_enc[FEAT_COLS].values.astype(np.float32)
    if fit_scaler:
        scaler = StandardScaler()
        X_cont = scaler.fit_transform(X_cont)
    else:
        X_cont = scaler.transform(X_cont)
    t = (
        torch.from_numpy(X_cont),
        torch.from_numpy(cat_codes['driver']),
        torch.from_numpy(cat_codes['race']),
        torch.from_numpy(cat_codes['compound']),
        torch.from_numpy(cat_codes['year']),
    )
    return t, scaler


def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    for batch in loader:
        x_cont, x_dr, x_rc, x_cmp, x_yr, y = [b.to(device) for b in batch]
        optimizer.zero_grad()
        logits = model(x_cont, x_dr, x_rc, x_cmp, x_yr)
        loss   = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def predict(model, tensors):
    """Inference in large batches, returns numpy probabilities."""
    model.eval()
    t_cont, t_dr, t_rc, t_cmp, t_yr = [t.to(device) for t in tensors]
    preds, n = [], t_cont.size(0)
    inf_bs = BATCH_SIZE * 4  # larger batch at inference (no gradients)
    for s in range(0, n, inf_bs):
        e = min(s + inf_bs, n)
        logits = model(t_cont[s:e], t_dr[s:e], t_rc[s:e], t_cmp[s:e], t_yr[s:e])
        preds.append(torch.sigmoid(logits).cpu().numpy())
    return np.concatenate(preds)

Batch size  : 4096
Max epochs  : 100  (early stop patience=10)
LR / WD     : 0.001 / 0.0001
Positive rate: 0.199  →  pos_weight: 4.03


## 5-Fold GroupKFold CV

Same GroupKFold by Race+Year as all other notebooks. For each fold:
1. Compute fold-aware target encodings (train rows only → val/test)
2. Fit `StandardScaler` on training fold continuous features
3. Train MLP with AdamW + CosineAnnealingLR, early stopping on val AUC
4. Save best model weights (`.pt`) and generate OOF + test predictions

In [7]:
n_train      = len(train)
oof_preds    = np.zeros(n_train, dtype=np.float64)
test_fold_preds = []
fold_aucs    = []
fold_best_ep = []

criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([POS_WEIGHT_VAL], dtype=torch.float32).to(device)
)

print(f'Training 5-fold MLP — device={device}  batch={BATCH_SIZE}  max_epochs={MAX_EPOCHS}')
print('=' * 72)

for fold_idx in range(5):
    t0 = time.time()
    torch.manual_seed(SEED + fold_idx)
    np.random.seed(SEED + fold_idx)

    tr_idx  = np.where(train['fold'] != fold_idx)[0]
    val_idx = np.where(train['fold'] == fold_idx)[0]

    # Fold-aware target encodings
    train_enc = apply_target_encodings(train.iloc[tr_idx], train.iloc[tr_idx])
    val_enc   = apply_target_encodings(train.iloc[tr_idx], train.iloc[val_idx])
    test_enc  = apply_target_encodings(train.iloc[tr_idx], test)

    # Categorical label codes
    cat_tr  = encode_cats(train.iloc[tr_idx])
    cat_val = encode_cats(train.iloc[val_idx])
    cat_te  = encode_cats(test)

    # Tensors (scaler fit on train fold only)
    tensors_tr,  scaler = df_to_tensors(train_enc, cat_tr,  fit_scaler=True)
    tensors_val, _      = df_to_tensors(val_enc,   cat_val, scaler=scaler)
    tensors_te,  _      = df_to_tensors(test_enc,  cat_te,  scaler=scaler)

    y_tr  = torch.tensor(train_enc['PitNextLap'].values, dtype=torch.float32)
    y_val = val_enc['PitNextLap'].values

    # DataLoader (shuffle train, drop last incomplete batch for stable BN)
    ds_tr = TensorDataset(*tensors_tr, y_tr)
    dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=0, drop_last=True)

    model     = make_model()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)

    best_auc   = 0.0
    best_ep    = 0
    best_state = None
    no_improve = 0

    for ep in range(MAX_EPOCHS):
        loss  = train_one_epoch(model, dl_tr, optimizer, criterion)
        scheduler.step()

        val_p  = predict(model, tensors_val)
        ep_auc = roc_auc_score(y_val, val_p)

        if ep_auc > best_auc + 1e-5:
            best_auc   = ep_auc
            best_ep    = ep + 1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1

        if (ep + 1) % 5 == 0:
            lr_now = scheduler.get_last_lr()[0]
            print(f'  Fold {fold_idx} ep {ep+1:3d}: loss={loss:.4f}  val_auc={ep_auc:.4f}  '
                  f'best={best_auc:.4f}@ep{best_ep}  lr={lr_now:.2e}')

        if no_improve >= PATIENCE:
            print(f'  Fold {fold_idx}: early stop at ep {ep+1}  (best was ep {best_ep}, AUC={best_auc:.4f})')
            break

    # Restore best weights
    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

    oof_preds[val_idx] = predict(model, tensors_val)
    fold_auc = roc_auc_score(y_val, oof_preds[val_idx])
    fold_aucs.append(fold_auc)
    fold_best_ep.append(best_ep)

    test_fold_preds.append(predict(model, tensors_te))

    torch.save(best_state, models_dir / f'mlp_fold{fold_idx}.pt')

    elapsed = time.time() - t0
    print(f'  ─── Fold {fold_idx} DONE: AUC={fold_auc:.4f}  best_epoch={best_ep}  ({elapsed/60:.1f} min)')
    print()

test_preds = np.mean(test_fold_preds, axis=0)
oof_auc    = roc_auc_score(train['PitNextLap'], oof_preds)

print('=' * 72)
print(f'OOF AUC: {oof_auc:.4f}  |  fold std: {np.std(fold_aucs):.4f}')
print(f'Fold AUCs   : {[f"{a:.4f}" for a in fold_aucs]}')
print(f'Best epochs : {fold_best_ep}')

Training 5-fold MLP — device=cpu  batch=4096  max_epochs=100
  Fold 0 ep   5: loss=0.6166  val_auc=0.8978  best=0.8978@ep5  lr=9.94e-04
  Fold 0 ep  10: loss=0.5608  val_auc=0.9038  best=0.9038@ep10  lr=9.76e-04
  Fold 0 ep  15: loss=0.5422  val_auc=0.9000  best=0.9040@ep11  lr=9.46e-04
  Fold 0 ep  20: loss=0.5300  val_auc=0.9020  best=0.9040@ep11  lr=9.05e-04
  Fold 0: early stop at ep 21  (best was ep 11, AUC=0.9040)
  ─── Fold 0 DONE: AUC=0.9040  best_epoch=11  (8.1 min)

  Fold 1 ep   5: loss=0.6445  val_auc=0.8634  best=0.8634@ep5  lr=9.94e-04
  Fold 1 ep  10: loss=0.5766  val_auc=0.8933  best=0.8933@ep10  lr=9.76e-04
  Fold 1 ep  15: loss=0.5471  val_auc=0.9027  best=0.9035@ep14  lr=9.46e-04
  Fold 1 ep  20: loss=0.5294  val_auc=0.9056  best=0.9073@ep19  lr=9.05e-04
  Fold 1 ep  25: loss=0.5157  val_auc=0.9068  best=0.9073@ep19  lr=8.54e-04
  Fold 1 ep  30: loss=0.5052  val_auc=0.9065  best=0.9074@ep29  lr=7.94e-04
  Fold 1 ep  35: loss=0.4954  val_auc=0.9043  best=0.9074@ep29  

## Correlation Analysis vs LGBM NB12

Load `lgbm_tuned_oof` from NB12 and compute Spearman ρ. This is the key diagnostic:
- ρ < 0.90 → meaningful diversity; strong case for rank-average inclusion
- 0.90–0.95 → marginal diversity; include only if rank-avg Δ ≥ +0.001
- ρ ≥ 0.95 → same error pattern as GBMs; no benefit

In [8]:
# Load LGBM NB12 and CB NB11 OOF — align by id
lgbm_df = pd.read_parquet(processed_dir / 'oof_predictions_nb12.parquet')
cb_df   = pd.read_parquet(processed_dir / 'oof_predictions_nb11.parquet')

lgbm_df = lgbm_df.sort_values('id').reset_index(drop=True)
cb_df   = cb_df.sort_values('id').reset_index(drop=True)

train_s = train.sort_values('id').reset_index(drop=True)
# Reorder oof_preds to match id-sorted train
id_to_pos = {row_id: pos for pos, row_id in enumerate(train['id'].values)}
oof_sorted = oof_preds[[id_to_pos[i] for i in train_s['id'].values]]

lgbm_oof = lgbm_df['lgbm_tuned_oof'].values
cb_oof   = cb_df['cb_plain_oof'].values
y_true   = train_s['PitNextLap'].values

rho_lgbm, _ = spearmanr(lgbm_oof, oof_sorted)
rho_cb,   _ = spearmanr(cb_oof,   oof_sorted)
rho_lb,   _ = spearmanr(lgbm_oof, cb_oof)

AUC_GATE = 0.895

print('=' * 72)
print('MLP vs GBM Correlation Analysis')
print('=' * 72)
print(f'  MLP solo AUC      : {oof_auc:.4f}')
print(f'  LGBM NB12 AUC     : 0.9024  (reference)')
print(f'  CatBoost NB11 AUC : 0.9016  (reference)')
print()
print(f'  Spearman ρ  MLP vs LGBM : {rho_lgbm:.4f}  (target < 0.90 for meaningful diversity)')
print(f'  Spearman ρ  MLP vs CB   : {rho_cb:.4f}')
print(f'  Spearman ρ  LGBM vs CB  : {rho_lb:.4f}  (NB13 reference — ρ=0.969)')
print()

in_auc_gate    = oof_auc >= AUC_GATE
is_diverse     = rho_lgbm < 0.90
is_marginal    = 0.90 <= rho_lgbm < 0.95

print('Gate Check:')
print(f'  AUC ≥ {AUC_GATE}: {"PASS" if in_auc_gate else "FAIL"}  ({oof_auc:.4f})')
print(f'  ρ < 0.90  : {"PASS — strong diversity" if is_diverse else ("marginal" if is_marginal else "FAIL — correlated")}')
print()
if in_auc_gate and is_diverse:
    print('→ INCLUDE in rank average: both AUC and diversity gates passed.')
elif in_auc_gate and is_marginal:
    print('→ MARGINAL: AUC passes but ρ is 0.90–0.95. Include only if rank-avg Δ ≥ +0.001.')
elif in_auc_gate:
    print('→ BORDERLINE EXCLUDE: AUC passes but ρ ≥ 0.95 — same errors as GBMs. Rank-avg unlikely to help.')
else:
    print('→ EXCLUDE: MLP AUC below gate. Too weak for ensemble.')

MLP vs GBM Correlation Analysis
  MLP solo AUC      : 0.9102
  LGBM NB12 AUC     : 0.9024  (reference)
  CatBoost NB11 AUC : 0.9016  (reference)

  Spearman ρ  MLP vs LGBM : 0.7910  (target < 0.90 for meaningful diversity)
  Spearman ρ  MLP vs CB   : 0.8091
  Spearman ρ  LGBM vs CB  : 0.9687  (NB13 reference — ρ=0.969)

Gate Check:
  AUC ≥ 0.895: PASS  (0.9102)
  ρ < 0.90  : PASS — strong diversity

→ INCLUDE in rank average: both AUC and diversity gates passed.


In [9]:
# Rank-average ensemble evaluation — compare all combinations on OOF
def rank_norm(arr):
    return rankdata(arr) / len(arr)

r_lgbm = rank_norm(lgbm_oof)
r_cb   = rank_norm(cb_oof)
r_mlp  = rank_norm(oof_sorted)

auc_lgbm_solo = roc_auc_score(y_true, lgbm_oof)
auc_cb_solo   = roc_auc_score(y_true, cb_oof)
auc_mlp_solo  = roc_auc_score(y_true, oof_sorted)
auc_rank_2m   = roc_auc_score(y_true, (r_lgbm + r_cb) / 2)
auc_rank_3m   = roc_auc_score(y_true, (r_lgbm + r_cb + r_mlp) / 3)
auc_mlp_cb    = roc_auc_score(y_true, (r_mlp + r_cb) / 2)
auc_mlp_lgbm  = roc_auc_score(y_true, (r_mlp + r_lgbm) / 2)

print('Rank-Average Ensemble Comparison')
print('=' * 72)
print(f'  LGBM NB12 solo        : {auc_lgbm_solo:.4f}  (NB12 reference)')
print(f'  CatBoost NB11 solo    : {auc_cb_solo:.4f}  (NB11 reference)')
print(f'  MLP solo              : {auc_mlp_solo:.4f}')
print()
print(f'  Rank avg LGBM+CB      : {auc_rank_2m:.4f}  (current best — v002 submission CV)')
print(f'  Rank avg LGBM+CB+MLP  : {auc_rank_3m:.4f}  Δ vs 2-model: {auc_rank_3m - auc_rank_2m:+.4f}')
print(f'  Rank avg MLP+CB       : {auc_mlp_cb:.4f}  Δ vs 2-model: {auc_mlp_cb - auc_rank_2m:+.4f}')
print(f'  Rank avg MLP+LGBM     : {auc_mlp_lgbm:.4f}  Δ vs 2-model: {auc_mlp_lgbm - auc_rank_2m:+.4f}')
print()

best_auc_val = max(auc_rank_2m, auc_rank_3m, auc_mlp_cb, auc_mlp_lgbm)
best_name    = ['LGBM+CB', 'LGBM+CB+MLP', 'MLP+CB', 'MLP+LGBM'][
    np.argmax([auc_rank_2m, auc_rank_3m, auc_mlp_cb, auc_mlp_lgbm])]
print(f'Best combination : {best_name}  →  CV AUC {best_auc_val:.4f}')
print(f'Projected LB AUC : {best_auc_val + 0.025:.4f}  (using +0.025 CV→LB boost)')

Rank-Average Ensemble Comparison
  LGBM NB12 solo        : 0.9024  (NB12 reference)
  CatBoost NB11 solo    : 0.9016  (NB11 reference)
  MLP solo              : 0.9102

  Rank avg LGBM+CB      : 0.9032  (current best — v002 submission CV)
  Rank avg LGBM+CB+MLP  : 0.9134  Δ vs 2-model: +0.0102
  Rank avg MLP+CB       : 0.9145  Δ vs 2-model: +0.0114
  Rank avg MLP+LGBM     : 0.9150  Δ vs 2-model: +0.0118

Best combination : MLP+LGBM  →  CV AUC 0.9150
Projected LB AUC : 0.9400  (using +0.025 CV→LB boost)


In [10]:
# Save OOF predictions (row order matches original train)
oof_df = pd.DataFrame({
    'id':         train['id'].values,
    'fold':       train['fold'].values,
    'PitNextLap': train['PitNextLap'].values,
    'mlp_oof':    oof_preds,
})
oof_path = processed_dir / 'oof_predictions_nb15.parquet'
oof_df.to_parquet(oof_path, index=False)
print(f'Saved OOF : {oof_path}  shape={oof_df.shape}')

# Save test predictions
test_df_out = pd.DataFrame({'id': test['id'].values, 'mlp_pred': test_preds})
test_path = processed_dir / 'test_predictions_nb15.parquet'
test_df_out.to_parquet(test_path, index=False)
print(f'Saved test: {test_path}  shape={test_df_out.shape}')

# Save summary for NB17
summary = {
    'mlp_oof_auc':          float(oof_auc),
    'fold_aucs':            [float(a) for a in fold_aucs],
    'fold_std':             float(np.std(fold_aucs)),
    'fold_best_epochs':     fold_best_ep,
    'spearman_rho_lgbm':    float(rho_lgbm),
    'spearman_rho_cb':      float(rho_cb),
    'auc_rank2m_lgbm_cb':   float(auc_rank_2m),
    'auc_rank3m_all':       float(auc_rank_3m),
    'auc_rank2m_mlp_cb':    float(auc_mlp_cb),
    'auc_rank2m_mlp_lgbm':  float(auc_mlp_lgbm),
    'in_auc_gate':          bool(in_auc_gate),
    'is_diverse':           bool(is_diverse),
    'best_ensemble_name':   best_name,
    'best_ensemble_auc':    float(best_auc_val),
    'model_config': {
        'n_cont':        len(FEAT_COLS),
        'emb_dims':      {'driver': EMB_DIM_DRIVER, 'race': EMB_DIM_RACE,
                          'compound': EMB_DIM_COMPOUND, 'year': EMB_DIM_YEAR},
        'hidden':        [256, 128, 64],
        'dropout':       [0.3, 0.2, 0.1],
        'batch_size':    BATCH_SIZE,
        'max_epochs':    MAX_EPOCHS,
        'patience':      PATIENCE,
        'lr':            LR,
        'weight_decay':  WEIGHT_DECAY,
        'pos_weight':    float(POS_WEIGHT_VAL),
    },
}
summary_path = models_dir / 'nb15_summary.pkl'
with open(summary_path, 'wb') as f:
    pickle.dump(summary, f)
print(f'Saved summary: {summary_path}')

print()
print('─' * 72)
print('NEXT STEPS (NB16 / NB17):')
print(f'  MLP solo AUC   : {oof_auc:.4f}  ({"≥" if in_auc_gate else "<"} 0.895 gate → {"PASS" if in_auc_gate else "FAIL"})')
print(f'  Best ensemble  : {best_name}  →  CV {best_auc_val:.4f}  →  proj LB {best_auc_val+0.025:.4f}')
print(f'  Decision       : {"Include MLP in NB17 rank avg" if in_auc_gate else "MLP excluded — NB17 uses LGBM+CB only"}')

Saved OOF : c:\Repos\predict-f1-pit-stops\data\processed\oof_predictions_nb15.parquet  shape=(439140, 4)
Saved test: c:\Repos\predict-f1-pit-stops\data\processed\test_predictions_nb15.parquet  shape=(188165, 2)
Saved summary: c:\Repos\predict-f1-pit-stops\models\nb15_summary.pkl

────────────────────────────────────────────────────────────────────────
NEXT STEPS (NB16 / NB17):
  MLP solo AUC   : 0.9102  (≥ 0.895 gate → PASS)
  Best ensemble  : MLP+LGBM  →  CV 0.9150  →  proj LB 0.9400
  Decision       : Include MLP in NB17 rank avg


## Summary

| Metric | Value |
|--------|-------|
| MLP solo OOF AUC | (see above) |
| Spearman ρ vs LGBM | (see above) |
| Best rank-avg ensemble | (see above) |
| AUC gate (≥0.895) | (see above) |

**Artifacts written:**
- `data/processed/oof_predictions_nb15.parquet` — columns: `id`, `fold`, `PitNextLap`, `mlp_oof`
- `data/processed/test_predictions_nb15.parquet` — columns: `id`, `mlp_pred` (5-fold averaged)
- `models/mlp_fold{0-4}.pt` — best-epoch weights for each fold
- `models/nb15_summary.pkl` — AUCs, ρ values, config, gate results for NB17

**NB17 ensemble inputs:**
- `oof_predictions_nb12.parquet` → `lgbm_tuned_oof` (0.9024)
- `oof_predictions_nb11.parquet` → `cb_plain_oof` (0.9016)
- `oof_predictions_nb15.parquet` → `mlp_oof` (this notebook)
- `test_predictions_nb12.parquet` → `lgbm_tuned_pred`
- `test_predictions_nb11.parquet` → `cb_plain_pred`
- `test_predictions_nb15.parquet` → `mlp_pred`